# Log Loss (対数損失) の概念を理解するチュートリアル

このノートブックでは、機械学習の二値分類タスクで最もよく使われる評価指標の一つである **Log Loss (対数損失 / 二値交差エントロピー損失)** について学びます。

## 目次
1. Log Loss とは？
2. なぜ正解率 (Accuracy) だけでは足りないのか？
3. Log Loss の数式
4. グラフによる可視化
5. 具体的なシミュレーションと計算例
6. scikit-learn を使った計算

## 準備

可視化と計算のために `numpy`, `matplotlib`, `scikit-learn` を使用します。
※ パッケージ管理には `uv` を使用します。

```bash
uv pip install numpy matplotlib scikit-learn
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import log_loss

## 1. Log Loss とは？

**Log Loss (対数損失)** は、分類モデルが予測した**「確率」の正しさを測るための指標**です。値が **0 に近いほど優れた予測**であることを示します。

正解率 (Accuracy) は予測された「クラス (0か1か)」の正誤のみを判定しますが、Log Loss は**「どれくらいの自信を持って (確率何%で) 予測したか」**を評価します。

- 予測が正解に近く、かつ確信度（確率）が高いほど、Log Loss は低くなります。
- 予測が不正解で、かつその不正解に対して自信満々（確率が極端）であるほど、Log Loss は非常に大きくなります。

## 2. Log Loss の数式

二値分類における1データあたりの Log Loss は以下の式で定義されます。

$$ \text{Log Loss} = - (y \log(p) + (1 - y) \log(1 - p)) $$

- $y$: 実際のラベル (0 または 1)
- $p$: モデルが「ラベルが 1 である」と予測した確率 ($0 \le p \le 1$)
- $\log$: 自然対数 (底が $e$)

データセット全体 ($N$ 個のデータ) の平均 Log Loss は以下のようになります。

$$ \text{Mean Log Loss} = -\frac{1}{N} \sum_{i=1}^{N} [y_i \log(p_i) + (1 - y_i) \log(1 - p_i)] $$

## 3. グラフによる可視化

実際のラベルが $y = 1$ の場合と $y = 0$ の場合において、予測確率 $p$ に応じて損失がどのように変化するかをグラフで確認してみましょう。

In [ ]:
# 予測確率 p を 0.001 から 0.999 まで生成 (log(0) による無限大を避けるため)
p = np.linspace(0.001, 0.999, 100)

# y = 1 のときの損失: -log(p)
loss_y1 = -np.log(p)

# y = 0 のときの損失: -log(1 - p)
loss_y0 = -np.log(1 - p)

# プロットの作成
plt.figure(figsize=(10, 6))
plt.plot(p, loss_y1, label='Actual Label y = 1 (Loss = -log(p))', color='blue', linewidth=2)
plt.plot(p, loss_y0, label='Actual Label y = 0 (Loss = -log(1-p))', color='orange', linewidth=2)

plt.title('Log Loss vs Predicted Probability', fontsize=14)
plt.xlabel('Predicted Probability (p) for Class 1', fontsize=12)
plt.ylabel('Loss Value', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(fontsize=12)
plt.ylim(0, 5) # 見やすさのために縦軸の上限を制限

plt.show()

### グラフからわかる特徴
- 実際のラベルが **$y = 1$** のとき、予測確率 $p$ が $1$ に近づくほど損失は $0$ に近づきますが、$p$ が $0$ に近づくと損失は指数関数的に跳ね上がります。
- 実際のラベルが **$y = 0$** のとき、予測確率 $p$ が $0$ に近づくほど損失は $0$ に近づきますが、$p$ が $1$ に近づくと損失は跳ね上がります。
- つまり、**「自信満々に間違える」ことに対して非常に厳しいペナルティを課す**のが Log Loss の特徴です。

## 4. 具体的なシミュレーション例

いくつかの予測パターンを考えて、手動計算してみましょう。

例として、**実際のラベルが $y = 1$（陽性）**のときの異なる4つの予測パターンを比較します。

In [ ]:
def single_log_loss(y_actual, p_pred):
    # 極端な値でのlog(0)を避けるためのクリッピング
    p_pred = max(min(p_pred, 0.9999999), 0.0000001)
    return -(y_actual * np.log(p_pred) + (1 - y_actual) * np.log(1 - p_pred))

y_actual = 1
scenarios = {
    "パターンA (自信のある正しい予測)": 0.95,
    "パターンB (自信のない正しい予測)": 0.60,
    "パターンC (自信のない誤った予測)": 0.40,
    "パターンD (自信のある大誤診)": 0.01
}

print(f"実際のラベル y = {y_actual} のとき:\n")
for name, p_pred in scenarios.items():
    loss = single_log_loss(y_actual, p_pred)
    # 正解率的な見方での分類結果 (0.5しきい値)
    decision = "正解" if (p_pred >= 0.5) else "不正解"
    print(f"- {name}: 予測確率={p_pred:.2f} ({decision}) => Loss = {loss:.4f}")

### シミュレーション結果の考察
- **パターンB (0.60)** と **パターンC (0.40)** は、しきい値 $0.5$ で見れば単に「正解」と「不正解」に分かれますが、Log Loss で見るとそれぞれ **0.5108** と **0.9163** であり、そこまで極端な差はありません。
- 一方、**パターンD (0.01)** は「絶対に 0（陰性）である」と自信満々に予測して外れたため、Loss は **4.6052** となり、他のパターンの数倍〜数十倍の大きなペナルティを受けています。
- この性質があるため、モデルのトレーニング時に Log Loss を最小化しようとすると、モデルは単に正答数を増やすだけでなく、**「自信の度合い（確率）」を正しくキャリブレーションする**ようになります。

## 5. scikit-learn を使った複数データの Log Loss 計算

実際のデータセット全体に対する平均 Log Loss を `sklearn.metrics.log_loss` を使って計算します。

In [ ]:
# 実際のラベル (5データ分)
y_true = np.array([1, 0, 1, 1, 0])

# モデルAの予測確率 (全体的に自信があり、精度も高い)
y_pred_A = np.array([0.9, 0.1, 0.8, 0.85, 0.2])

# モデルBの予測確率 (正解はしているが自信がない、または一部で自信満々に外している)
y_pred_B = np.array([0.6, 0.4, 0.7, 0.99, 0.9]) # 最後のデータ(正解=0)に確率0.9と大誤診している

# それぞれの正解率(Accuracy)の計算 (しきい値0.5)
acc_A = np.mean((y_pred_A >= 0.5) == y_true)
acc_B = np.mean((y_pred_B >= 0.5) == y_true)

# Log Loss の計算
loss_A = log_loss(y_true, y_pred_A)
loss_B = log_loss(y_true, y_pred_B)

print(f"【モデルA】")
print(f"  予測確率: {y_pred_A}")
print(f"  正解率  : {acc_A * 100:.1f}%")
print(f"  Log Loss: {loss_A:.4f}")
print()
print(f"【モデルB】")
print(f"  予測確率: {y_pred_B}")
print(f"  正解率  : {acc_B * 100:.1f}%")
print(f"  Log Loss: {loss_B:.4f}")

### まとめ
- モデルAとモデルBはどちらも5問中4問正解しており、**正解率 (Accuracy) は 80.0% で同じ**です。
- しかし、モデルBは最後のデータ（実際のラベルが 0）に対して確率 0.9 で 1 であると予測して外したため、Log Loss が非常に大きくなっています。
- 分類モデルの評価においては、正解率だけでなく **Log Loss** のような確率ベースの指標を合わせて見ることで、モデルの予測の質や安定性をより深く理解することができます。